# Outlier Threshold Comparison — P95 vs P99.5
**Method 3b | Per Building Type**

P95 was tested instead of P99.5 for outlier removal.
The concern is that large outlier buildings (likely misclassified industrial/farm buildings)
occupy one entire cluster and reduce accuracy for the remaining 99% of buildings.

**Current approach (P99.5):** removes top 0.5% per building type — ~20,669 buildings
**P95 approach:** removes top 5% per building type — more aggressive

This notebook compares:
- How many buildings are removed at each threshold
- Whether clustering results change meaningfully
- Whether the large outlier cluster disappears at P95

## Step 1 — Imports and Configuration

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

BASE_DIR   = os.getcwd()
FILE_M3B   = os.path.join(BASE_DIR, 'DEA_method3b_final.parquet')
OUTPUT_DIR = os.path.join(BASE_DIR, 'clustering_results', 'outlier_comparison')
os.makedirs(OUTPUT_DIR, exist_ok=True)

SIZE_CLASSES = ['SFH', 'TH', 'MFH', 'AB']
RANDOM_STATE = 42

print('Ready')

## Step 2 — Load Data

In [ ]:
m3b = pd.read_parquet(FILE_M3B, columns=[
    'id', 'size_class', 'area_m2',
    'elec_m3b_kwh', 'heat_total_kwh', 'pv_potential_kwh'
])
print(f'Buildings loaded: {len(m3b):,}')
print(f'\nCount per building type:')
print(m3b['size_class'].value_counts().to_string())

## Step 3 — Compare Removal at Different Thresholds

Check how many buildings are removed at P90, P95, P99, P99.5 per building type.
This shows where the outliers are concentrated.

In [ ]:
thresholds_to_test = [0.90, 0.95, 0.99, 0.995]

print('Buildings removed per threshold:')
print(f'  {"Threshold":<12} {"SFH":>10} {"TH":>10} {"MFH":>10} {"AB":>10} {"TOTAL":>10}')
print('-' * 65)

for thr in thresholds_to_test:
    removed_per_sc = {}
    for sc in SIZE_CLASSES:
        mask = m3b['size_class'] == sc
        p = m3b.loc[mask, 'area_m2'].quantile(thr)
        n_removed = (m3b.loc[mask, 'area_m2'] > p).sum()
        removed_per_sc[sc] = n_removed
    total = sum(removed_per_sc.values())
    pct = total / len(m3b) * 100
    print(f'  P{thr*100:<9.1f} '
          f'{removed_per_sc["SFH"]:>10,} '
          f'{removed_per_sc["TH"]:>10,} '
          f'{removed_per_sc["MFH"]:>10,} '
          f'{removed_per_sc["AB"]:>10,} '
          f'{total:>10,}  ({pct:.2f}%)')

## Step 4 — Apply Both Thresholds and Compare Distributions

Apply P99.5 and P95 separately and compare the resulting heat demand distributions.
If P95 removes the outlier cluster, the upper tail of the distribution should be cut off.

In [ ]:
def apply_threshold(df, percentile):
    keep = pd.Series(True, index=df.index)
    thresholds = {}
    for sc in SIZE_CLASSES:
        mask = df['size_class'] == sc
        p = df.loc[mask, 'area_m2'].quantile(percentile)
        thresholds[sc] = p
        keep &= ~((df['size_class'] == sc) & (df['area_m2'] > p))
    return df[keep].copy(), thresholds

m3b_p995, thr_p995 = apply_threshold(m3b, 0.995)
m3b_p95,  thr_p95  = apply_threshold(m3b, 0.95)

print(f'P99.5 retained: {len(m3b_p995):,}  (removed {len(m3b)-len(m3b_p995):,}  = {(len(m3b)-len(m3b_p995))/len(m3b)*100:.2f}%)')
print(f'P95   retained: {len(m3b_p95):,}  (removed {len(m3b)-len(m3b_p95):,}  = {(len(m3b)-len(m3b_p95))/len(m3b)*100:.2f}%)')

# Compare thresholds
print('\nArea thresholds (m2):')
print(f'  {"Type":<6} {"P99.5":>10} {"P95":>10}  Difference')
for sc in SIZE_CLASSES:
    diff = thr_p995[sc] - thr_p95[sc]
    print(f'  {sc:<6} {thr_p995[sc]:>10.1f} {thr_p95[sc]:>10.1f}  -{diff:.1f} m2')

In [ ]:
# Compare heat demand distributions
fig, axes = plt.subplots(2, 4, figsize=(18, 8))

for col, sc in enumerate(SIZE_CLASSES):
    sub_p995 = m3b_p995[m3b_p995['size_class'] == sc]['heat_total_kwh']
    sub_p95  = m3b_p95[m3b_p95['size_class'] == sc]['heat_total_kwh']

    cap = sub_p995.quantile(0.99)

    # Top row: histogram comparison
    ax = axes[0][col]
    ax.hist(sub_p995.clip(upper=cap), bins=50, alpha=0.6,
            color='steelblue', label=f'P99.5  n={len(sub_p995):,}')
    ax.hist(sub_p95.clip(upper=cap),  bins=50, alpha=0.6,
            color='tomato',   label=f'P95    n={len(sub_p95):,}')
    ax.set_title(f'{sc} — Heat Demand Distribution', fontsize=9, fontweight='bold')
    ax.set_xlabel('Heat demand (kWh/yr)', fontsize=8)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

    # Bottom row: boxplot comparison
    ax2 = axes[1][col]
    bp = ax2.boxplot([sub_p995.values, sub_p95.values],
                     patch_artist=True,
                     medianprops={'color': 'black', 'linewidth': 2})
    bp['boxes'][0].set_facecolor('steelblue'); bp['boxes'][0].set_alpha(0.7)
    bp['boxes'][1].set_facecolor('tomato');    bp['boxes'][1].set_alpha(0.7)
    ax2.set_xticklabels(['P99.5', 'P95'], fontsize=9)
    ax2.set_title(f'{sc} — Boxplot Comparison', fontsize=9, fontweight='bold')
    ax2.set_ylabel('Heat demand (kWh/yr)', fontsize=8)
    ax2.grid(True, axis='y', alpha=0.3)

plt.suptitle('Heat Demand Distribution: P99.5 vs P95 Outlier Removal',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'outlier_comparison_distributions.pdf'),
            bbox_inches='tight')
plt.show()
print('Plot saved')

## Step 5 — Compare Clustering Results

Run K-Means with k=4 on heat demand for both thresholds.
Check if cluster boundaries shift significantly between P99.5 and P95.

In [ ]:
def cluster_heat(df, label, k=4):
    results = {}
    for sc in SIZE_CLASSES:
        subset = df[df['size_class'] == sc].copy()
        vals = subset['heat_total_kwh'].values.astype(float)

        cap  = np.percentile(vals, 99.9)
        vals_clipped = np.clip(vals, 0, cap)
        X = StandardScaler().fit_transform(vals_clipped.reshape(-1, 1))

        km = MiniBatchKMeans(n_clusters=k, random_state=RANDOM_STATE,
                             n_init=10, batch_size=5_000)
        labels = km.fit_predict(X)

        # Relabel 0 = lowest
        order = sorted(range(k), key=lambda c: vals[labels == c].mean())
        remap = {old: new for new, old in enumerate(order)}
        labels = np.array([remap[c] for c in labels])

        stats = []
        for c in range(k):
            v = vals[labels == c]
            stats.append({
                'size_class': sc, 'cluster': c,
                'n': int(len(v)),
                'mean': round(float(v.mean()), 0),
                'median': round(float(np.median(v)), 0),
                'max': round(float(v.max()), 0),
            })
        results[sc] = pd.DataFrame(stats)
    return results

print('Clustering with P99.5...')
results_p995 = cluster_heat(m3b_p995, 'P99.5')
print('Clustering with P95...')
results_p95  = cluster_heat(m3b_p95,  'P95')

print('\nCLUSTER MEAN COMPARISON (heat demand kWh/yr):')
print(f'  {"Type":<6} {"Cluster":<10} {"P99.5 mean":>12} {"P95 mean":>12} {"Difference":>12}')
print('-' * 60)
for sc in SIZE_CLASSES:
    for c in range(4):
        m_p995 = results_p995[sc].loc[results_p995[sc]['cluster']==c, 'mean'].values[0]
        m_p95  = results_p95[sc].loc[results_p95[sc]['cluster']==c,  'mean'].values[0]
        diff   = m_p95 - m_p995
        pct    = diff / m_p995 * 100 if m_p995 > 0 else 0
        flag   = '  <- SIGNIFICANT' if abs(pct) > 10 else ''
        print(f'  {sc:<6} C{c:<9} {m_p995:>12,.0f} {m_p95:>12,.0f} '
              f'{diff:>+12,.0f}  ({pct:+.1f}%){flag}')

## Step 6 — Summary and Recommendation

In [ ]:
print('SUMMARY')
print('='*60)
print()
print(f'  P99.5 retained: {len(m3b_p995):,} buildings ({len(m3b_p995)/len(m3b)*100:.1f}%)')
print(f'  P95   retained: {len(m3b_p95):,} buildings ({len(m3b_p95)/len(m3b)*100:.1f}%)')
print()

# Check if cluster means changed significantly
significant_changes = 0
for sc in SIZE_CLASSES:
    for c in range(4):
        m_p995 = results_p995[sc].loc[results_p995[sc]['cluster']==c, 'mean'].values[0]
        m_p95  = results_p95[sc].loc[results_p95[sc]['cluster']==c,  'mean'].values[0]
        pct = abs(m_p95 - m_p995) / m_p995 * 100 if m_p995 > 0 else 0
        if pct > 10:
            significant_changes += 1

print(f'  Cluster means changed >10%: {significant_changes} out of {len(SIZE_CLASSES)*4}')
print()

if significant_changes > 4:
    print('  RECOMMENDATION: P95 changes clustering significantly.')
    print('  Consider whether removing 5% is justified based on results.')
else:
    print('  RECOMMENDATION: P95 does not change clustering significantly.')
    print('  Safe to use P95 if it removes the outlier cluster.')
print()
print('  Results saved to: clustering_results/outlier_comparison/')